# 🔍 Fraud Detection — End-to-End Machine Learning Pipeline

**Tugas UTS Machine Learning — Fraud Detection (Klasifikasi)**

---

## 📌 Daftar Isi
1. [Setup & Import Libraries](#1)
2. [Load Dataset](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Data Preprocessing](#4)
5. [Handle Class Imbalance (SMOTE)](#5)
6. [Feature Engineering & Selection](#6)
7. [Model Training (Baseline)](#7)
8. [Hyperparameter Tuning dengan Optuna](#8)
9. [Evaluasi Model](#9)
10. [MLFlow Tracking](#10)
11. [Kesimpulan](#11)

---

### 🎯 Tujuan
Membangun pipeline end-to-end untuk mendeteksi transaksi **fraud** menggunakan dataset transaksi kartu kredit.
Target variabel: `Class` — 1 = Fraud, 0 = Normal

## 1. Setup & Import Libraries <a id='1'></a>

In [ ]:
# Install dependencies yang dibutuhkan
!pip install optuna mlflow imbalanced-learn scikit-learn pandas numpy matplotlib seaborn xgboost lightgbm --quiet

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline

# ── Imbalanced Learning ───────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# ── Boosting Models ───────────────────────────────────────────────
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ── Optuna (Hyperparameter Tuning) ────────────────────────────────
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── MLFlow (Experiment Tracking) ──────────────────────────────────
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print("✅ Semua library berhasil diimport!")
print(f"   Pandas     : {pd.__version__}")
print(f"   NumPy      : {np.__version__}")
print(f"   Optuna     : {optuna.__version__}")
print(f"   MLFlow     : {mlflow.__version__}")

## 2. Load Dataset <a id='2'></a>

Menggunakan dataset **Credit Card Fraud Detection** dari Kaggle (dataset publik yang sudah terkenal).

> 💡 **Cara ganti dataset**: Jika sudah punya `train_transaction.csv`, ubah bagian load data di bawah dan sesuaikan nama kolom target dari `Class` menjadi `isFraud`.

In [ ]:
# ── OPSI A: Download langsung dari OpenML (tidak perlu Kaggle API) ──
from sklearn.datasets import fetch_openml

print("⏳ Mengunduh dataset Credit Card Fraud dari OpenML...")
credit_card = fetch_openml(data_id=1597, as_frame=True, parser='auto')

df = credit_card.frame.copy()

# Pastikan kolom target bernama 'Class' dan bertipe integer
df['Class'] = df['Class'].astype(int)

print(f"✅ Dataset berhasil dimuat!")
print(f"   Jumlah baris   : {df.shape[0]:,}")
print(f"   Jumlah kolom   : {df.shape[1]}")
print(f"   Kolom target   : Class (0=Normal, 1=Fraud)")

In [ ]:
# ── Alternatif: Jika punya file lokal train_transaction.csv ─────────
# df = pd.read_csv('train_transaction.csv')
# df = df.rename(columns={'isFraud': 'Class'})  # sesuaikan nama target
# print(f"✅ Dataset lokal dimuat: {df.shape}")

In [ ]:
# Tampilkan 5 baris pertama
print("📋 Tampilan awal dataset:")
df.head()

In [ ]:
# Informasi umum dataset
print("📊 Informasi Dataset:")
df.info()

## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

In [ ]:
# ── Statistik Deskriptif ─────────────────────────────────────────
print("📈 Statistik Deskriptif:")
df.describe().T

In [ ]:
# ── Distribusi Kelas Target ───────────────────────────────────────
class_counts = df['Class'].value_counts()
class_pct    = df['Class'].value_counts(normalize=True) * 100

print("⚖️  Distribusi Kelas Target:")
print(f"   Normal (0) : {class_counts[0]:,} ({class_pct[0]:.2f}%)")
print(f"   Fraud  (1) : {class_counts[1]:,} ({class_pct[1]:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#2196F3', '#F44336']
axes[0].bar(['Normal (0)', 'Fraud (1)'], class_counts.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribusi Kelas Target', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Transaksi')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Normal (0)', 'Fraud (1)'],
            autopct='%1.2f%%', colors=colors, startangle=90,
            explode=(0, 0.1), shadow=True)
axes[1].set_title('Proporsi Kelas Target', fontsize=14, fontweight='bold')

plt.suptitle('Class Imbalance Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️  Dataset sangat tidak seimbang (imbalanced)! Perlu penanganan khusus.")

In [ ]:
# ── Cek Missing Values ────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print("✅ Tidak ada missing values!")
else:
    print(f"⚠️  Kolom dengan missing values:")
    print(missing_df)

In [ ]:
# ── Distribusi Fitur Amount & Time ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount by class
for cls, color, label in [(0, '#2196F3', 'Normal'), (1, '#F44336', 'Fraud')]:
    axes[0, 0].hist(df[df['Class']==cls]['Amount'], bins=50, alpha=0.7,
                    color=color, label=label, density=True)
axes[0, 0].set_title('Distribusi Amount per Kelas')
axes[0, 0].set_xlabel('Amount')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 500)

# Amount boxplot
df.boxplot(column='Amount', by='Class', ax=axes[0, 1], 
           boxprops=dict(color='navy'), medianprops=dict(color='red'))
axes[0, 1].set_title('Boxplot Amount per Kelas')
axes[0, 1].set_xlabel('Kelas (0=Normal, 1=Fraud)')

# Time distribution
for cls, color, label in [(0, '#2196F3', 'Normal'), (1, '#F44336', 'Fraud')]:
    axes[1, 0].hist(df[df['Class']==cls]['Time'], bins=50, alpha=0.7,
                    color=color, label=label, density=True)
axes[1, 0].set_title('Distribusi Time per Kelas')
axes[1, 0].set_xlabel('Time (seconds)')
axes[1, 0].legend()

# Correlation heatmap (subset)
sample_cols = ['V1', 'V2', 'V3', 'V4', 'V5', 'Amount', 'Class']
corr = df[sample_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1, 1], linewidths=0.5)
axes[1, 1].set_title('Correlation Heatmap (Subset Fitur)')

plt.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing <a id='4'></a>

Tahapan preprocessing yang akan dilakukan:
- Scaling fitur `Amount` dan `Time` menggunakan **RobustScaler** (tahan terhadap outlier)
- Train-test split dengan stratifikasi untuk menjaga proporsi kelas

In [ ]:
# ── Pisahkan Fitur dan Target ─────────────────────────────────────
X = df.drop(columns=['Class'])
y = df['Class']

print(f"Shape fitur (X) : {X.shape}")
print(f"Shape target (y): {y.shape}")
print(f"Kolom fitur     : {list(X.columns)}")

In [ ]:
# ── Scaling Amount & Time ─────────────────────────────────────────
# RobustScaler lebih tahan terhadap outlier dibanding StandardScaler
scaler = RobustScaler()

X_scaled = X.copy()
X_scaled[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

print("✅ Scaling selesai!")
print(f"   Amount sebelum — mean: {X['Amount'].mean():.2f}, std: {X['Amount'].std():.2f}")
print(f"   Amount sesudah  — mean: {X_scaled['Amount'].mean():.4f}, std: {X_scaled['Amount'].std():.4f}")

In [ ]:
# ── Train-Test Split (Stratified) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size    = 0.2,
    random_state = SEED,
    stratify     = y     # penting untuk data imbalanced!
)

print("✅ Split dataset selesai!")
print(f"   Train : {X_train.shape[0]:,} baris  | Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"   Test  : {X_test.shape[0]:,} baris   | Fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

## 5. Handle Class Imbalance dengan SMOTE <a id='5'></a>

Dataset ini sangat imbalanced (~0.17% fraud). Tanpa penanganan, model akan bias ke kelas mayoritas (Normal).

**SMOTE** (Synthetic Minority Oversampling Technique) membuat data sintetis untuk kelas minoritas (Fraud) agar proporsinya lebih seimbang.

In [ ]:
# ── Terapkan SMOTE pada data training saja ────────────────────────
# PENTING: SMOTE hanya diterapkan pada TRAIN set, bukan test set!

smote = SMOTE(sampling_strategy=0.1, random_state=SEED)
# sampling_strategy=0.1 artinya fraud akan menjadi 10% dari data normal

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("✅ SMOTE selesai diterapkan!")
print(f"   Sebelum SMOTE — Normal: {(y_train==0).sum():,} | Fraud: {(y_train==1).sum():,}")
print(f"   Sesudah SMOTE  — Normal: {(y_train_res==0).sum():,} | Fraud: {(y_train_res==1).sum():,}")
print(f"   Rasio Fraud setelah SMOTE: {y_train_res.mean()*100:.1f}%")

## 6. Feature Engineering & Selection <a id='6'></a>

In [ ]:
# ── Analisis Feature Importance menggunakan RandomForest cepat ────
from sklearn.ensemble import RandomForestClassifier

print("⏳ Menghitung feature importance (gunakan subset data untuk kecepatan)...")

# Gunakan subset kecil untuk kecepatan
idx_sample = np.random.choice(len(X_train_res), size=min(20000, len(X_train_res)), replace=False)
X_sample   = X_train_res.iloc[idx_sample]
y_sample   = y_train_res.iloc[idx_sample]

rf_quick = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)
rf_quick.fit(X_sample, y_sample)

feature_importance = pd.Series(
    rf_quick.feature_importances_,
    index=X_train_res.columns
).sort_values(ascending=False)

# Plot top 20 fitur
plt.figure(figsize=(12, 6))
colors_bar = plt.cm.RdYlGn(np.linspace(0.8, 0.2, 20))
feature_importance.head(20).plot(kind='bar', color=colors_bar, edgecolor='black', linewidth=0.5)
plt.title('Top 20 Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Fitur')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 10 fitur paling penting:")
print(feature_importance.head(10).to_string())

In [ ]:
# ── Pilih Top N Fitur ─────────────────────────────────────────────
TOP_N_FEATURES = 20
selected_features = feature_importance.head(TOP_N_FEATURES).index.tolist()

X_train_sel = X_train_res[selected_features]
X_test_sel  = X_test[selected_features]

print(f"✅ Fitur terpilih ({TOP_N_FEATURES} fitur): {selected_features}")

## 7. Model Training — Baseline <a id='7'></a>

Kita akan melatih beberapa model dari yang sederhana hingga kompleks:
1. **Logistic Regression** (baseline simpel)
2. **Decision Tree**
3. **Random Forest**
4. **XGBoost**
5. **LightGBM**

Metrik utama: **ROC-AUC** dan **F1-Score** (lebih representatif untuk imbalanced data)

In [ ]:
# ── Helper: Fungsi Evaluasi Model ────────────────────────────────
def evaluate_model(name, model, X_train, y_train, X_test, y_test, verbose=True):
    """Train dan evaluasi model, kembalikan dict hasil."""
    model.fit(X_train, y_train)
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]

    results = {
        'model'    : model,
        'name'     : name,
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall'   : recall_score(y_test, y_pred, zero_division=0),
        'f1'       : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_test, y_pred_prob),
        'avg_prec' : average_precision_score(y_test, y_pred_prob),
        'y_pred'   : y_pred,
        'y_prob'   : y_pred_prob,
    }

    if verbose:
        print(f"\n{'='*55}")
        print(f"  {name}")
        print(f"{'='*55}")
        print(f"  Accuracy  : {results['accuracy']:.4f}")
        print(f"  Precision : {results['precision']:.4f}")
        print(f"  Recall    : {results['recall']:.4f}")
        print(f"  F1-Score  : {results['f1']:.4f}")
        print(f"  ROC-AUC   : {results['roc_auc']:.4f}")
        print(f"  Avg Prec  : {results['avg_prec']:.4f}")

    return results

In [ ]:
# ── Training Semua Model Baseline ────────────────────────────────
baseline_models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=SEED, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, random_state=SEED, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=SEED,
        class_weight='balanced', n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=SEED, eval_metric='auc', verbosity=0,
        scale_pos_weight=(y_train==0).sum() / (y_train==1).sum()
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=SEED, class_weight='balanced', verbose=-1
    ),
}

all_results = {}
for name, model in baseline_models.items():
    print(f"⏳ Training {name}...")
    all_results[name] = evaluate_model(
        name, model, X_train_sel, y_train_res, X_test_sel, y_test
    )

print("\n✅ Semua model baseline selesai ditraining!")

In [ ]:
# ── Tabel Perbandingan Baseline ───────────────────────────────────
comparison_df = pd.DataFrame([
    {
        'Model'    : name,
        'Accuracy' : res['accuracy'],
        'Precision': res['precision'],
        'Recall'   : res['recall'],
        'F1-Score' : res['f1'],
        'ROC-AUC'  : res['roc_auc'],
        'Avg Prec' : res['avg_prec'],
    }
    for name, res in all_results.items()
]).set_index('Model').sort_values('ROC-AUC', ascending=False)

print("\n📊 Perbandingan Semua Model Baseline:")
print(comparison_df.to_string(float_format='{:.4f}'.format))

# Plot perbandingan
fig, ax = plt.subplots(figsize=(12, 6))
comparison_df[['F1-Score', 'ROC-AUC', 'Avg Prec']].plot(
    kind='bar', ax=ax, width=0.7,
    color=['#4CAF50', '#2196F3', '#FF9800'],
    edgecolor='black', linewidth=0.5
)
ax.set_title('Perbandingan Model Baseline', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_xticklabels(comparison_df.index, rotation=30, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='Target 0.9')
plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Hyperparameter Tuning dengan Optuna <a id='8'></a>

**Optuna** adalah framework hyperparameter optimization menggunakan **Bayesian Optimization**.

Kita akan tuning model terbaik (biasanya XGBoost atau LightGBM).

In [ ]:
# ── Optuna: Tuning LightGBM ───────────────────────────────────────

def objective_lgbm(trial):
    """
    Fungsi objective untuk Optuna.
    Setiap trial akan mencoba kombinasi hyperparameter yang berbeda.
    """
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'     : SEED,
        'class_weight'     : 'balanced',
        'verbose'          : -1,
    }

    model = LGBMClassifier(**params)

    # Cross-validation dengan StratifiedKFold
    skf   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    score = cross_val_score(
        model, X_train_sel, y_train_res,
        cv=skf, scoring='roc_auc', n_jobs=-1
    ).mean()

    return score


print("⏳ Menjalankan Optuna untuk LightGBM...")
print("   (30 trials — bisa disesuaikan lebih banyak untuk hasil lebih baik)")

study_lgbm = optuna.create_study(
    direction = 'maximize',
    sampler   = TPESampler(seed=SEED),
    study_name= 'lgbm_fraud_detection'
)

study_lgbm.optimize(
    objective_lgbm,
    n_trials  = 30,     # naikkan ke 100 untuk hasil lebih optimal
    timeout   = 300,    # max 5 menit
    show_progress_bar = True
)

print(f"\n✅ Optuna selesai!")
print(f"   Best ROC-AUC : {study_lgbm.best_value:.4f}")
print(f"   Best Params  :")
for k, v in study_lgbm.best_params.items():
    print(f"     {k}: {v}")

In [ ]:
# ── Visualisasi Optuna ────────────────────────────────────────────
try:
    from optuna.visualization import plot_optimization_history, plot_param_importances
    import plotly

    fig_history = plot_optimization_history(study_lgbm)
    fig_history.update_layout(title='Optuna: Optimization History (LightGBM)')
    fig_history.show()

    fig_params = plot_param_importances(study_lgbm)
    fig_params.update_layout(title='Optuna: Parameter Importances')
    fig_params.show()
except Exception as e:
    print(f"ℹ️  Plotly tidak tersedia untuk visualisasi Optuna: {e}")
    # Plot manual pakai matplotlib
    trials_df = study_lgbm.trials_dataframe()
    plt.figure(figsize=(10, 4))
    plt.plot(trials_df['number'], trials_df['value'], 'o-', color='#2196F3', alpha=0.7)
    plt.axhline(y=study_lgbm.best_value, color='red', linestyle='--', label=f'Best: {study_lgbm.best_value:.4f}')
    plt.title('Optuna Optimization History', fontweight='bold')
    plt.xlabel('Trial')
    plt.ylabel('ROC-AUC')
    plt.legend()
    plt.tight_layout()
    plt.savefig('optuna_history.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Train Model Final dengan Best Params ─────────────────────────
best_params_lgbm = study_lgbm.best_params.copy()
best_params_lgbm.update({'random_state': SEED, 'class_weight': 'balanced', 'verbose': -1})

best_lgbm = LGBMClassifier(**best_params_lgbm)

print("⏳ Training model final dengan best params...")
tuned_result = evaluate_model(
    'LightGBM (Tuned)', best_lgbm,
    X_train_sel, y_train_res,
    X_test_sel, y_test
)
all_results['LightGBM (Tuned)'] = tuned_result

## 9. Evaluasi Model — Detail <a id='9'></a>

In [ ]:
# ── Confusion Matrix untuk Model Terbaik ─────────────────────────
best_result = tuned_result

cm = confusion_matrix(y_test, best_result['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'],
            linewidths=1, linecolor='gray')
axes[0].set_title('Confusion Matrix — LightGBM (Tuned)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve semua model
for name, res in all_results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, lw=2, label=f"{name} (AUC={res['roc_auc']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Semua Model', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Precision-Recall Curve ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, res in all_results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    ax.plot(rec, prec, lw=2, label=f"{name} (AP={ap:.3f})")

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Semua Model', fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Classification Report Lengkap ────────────────────────────────
print("📋 Classification Report — LightGBM (Tuned):")
print(classification_report(
    y_test, best_result['y_pred'],
    target_names=['Normal (0)', 'Fraud (1)']
))

In [ ]:
# ── Tabel Rangkuman Akhir ─────────────────────────────────────────
final_df = pd.DataFrame([
    {
        'Model'    : name,
        'Accuracy' : f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall'   : f"{res['recall']:.4f}",
        'F1-Score' : f"{res['f1']:.4f}",
        'ROC-AUC'  : f"{res['roc_auc']:.4f}",
    }
    for name, res in all_results.items()
]).set_index('Model')

print("\n📊 Tabel Perbandingan Akhir Semua Model:")
print(final_df.to_string())

## 10. MLFlow Tracking <a id='10'></a>

**MLFlow** digunakan untuk mencatat dan membandingkan eksperimen secara sistematis.

Setiap run akan menyimpan: hyperparameter, metrik evaluasi, dan model artifact.

In [ ]:
# ── Setup MLFlow ──────────────────────────────────────────────────
# Set tracking URI (local folder)
mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('fraud-detection-uts')

print("✅ MLFlow siap digunakan!")
print("   Tracking URI : mlruns/")
print("   Experiment   : fraud-detection-uts")

In [ ]:
# ── Log semua model ke MLFlow ────────────────────────────────────
print("⏳ Logging semua model ke MLFlow...")

for name, res in all_results.items():
    with mlflow.start_run(run_name=name):

        # Log parameters
        model_params = res['model'].get_params()
        # Hanya log parameter yang serializable
        safe_params = {k: str(v) for k, v in model_params.items() if v is not None}
        mlflow.log_params(safe_params)

        # Log metrics
        mlflow.log_metric('accuracy',  res['accuracy'])
        mlflow.log_metric('precision', res['precision'])
        mlflow.log_metric('recall',    res['recall'])
        mlflow.log_metric('f1_score',  res['f1'])
        mlflow.log_metric('roc_auc',   res['roc_auc'])
        mlflow.log_metric('avg_precision', res['avg_prec'])

        # Log tags
        mlflow.set_tag('model_type', type(res['model']).__name__)
        mlflow.set_tag('dataset', 'credit_card_fraud')
        mlflow.set_tag('smote', 'True')
        mlflow.set_tag('feature_selection', f'top_{TOP_N_FEATURES}')

        # Log plots
        try:
            mlflow.log_artifact('class_distribution.png')
            mlflow.log_artifact('feature_importance.png')
        except:
            pass

        # Log model
        mlflow.sklearn.log_model(res['model'], artifact_path='model')

        print(f"   ✅ {name} berhasil dilog | ROC-AUC: {res['roc_auc']:.4f}")

print("\n🎉 Semua model berhasil dilog ke MLFlow!")
print("   Untuk melihat dashboard, jalankan di terminal:")
print("   $ mlflow ui --port 5000")
print("   Lalu buka browser: http://127.0.0.1:5000")

In [ ]:
# ── Tampilkan MLFlow Runs ─────────────────────────────────────────
runs = mlflow.search_runs(
    experiment_names=['fraud-detection-uts'],
    order_by=['metrics.roc_auc DESC']
)

display_cols = ['tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1_score',
                'metrics.roc_auc', 'metrics.precision', 'metrics.recall']

print("\n📋 MLFlow Experiment Runs (sorted by ROC-AUC):")
print(runs[display_cols].rename(columns={
    'tags.mlflow.runName': 'Model',
    'metrics.accuracy'  : 'Accuracy',
    'metrics.f1_score'  : 'F1-Score',
    'metrics.roc_auc'   : 'ROC-AUC',
    'metrics.precision' : 'Precision',
    'metrics.recall'    : 'Recall',
}).to_string(index=False, float_format='{:.4f}'.format))

## 11. Kesimpulan <a id='11'></a>

In [ ]:
# ── Rangkuman Eksperimen ──────────────────────────────────────────
best_model_name = max(all_results, key=lambda k: all_results[k]['roc_auc'])
best_res        = all_results[best_model_name]

print("=" * 60)
print("  📋 RANGKUMAN EKSPERIMEN FRAUD DETECTION")
print("=" * 60)
print(f"  Dataset           : Credit Card Fraud (OpenML)")
print(f"  Total data        : {len(df):,} transaksi")
print(f"  Fraud ratio       : {y.mean()*100:.2f}%")
print(f"  Handling imbalance: SMOTE (sampling_strategy=0.1)")
print(f"  Feature selection : Top {TOP_N_FEATURES} dari Random Forest")
print(f"  Hyperparameter opt: Optuna (30 trials, TPE Sampler)")
print(f"  Tracking          : MLFlow (experiment: fraud-detection-uts)")
print()
print(f"  🏆 Model Terbaik  : {best_model_name}")
print(f"     Accuracy       : {best_res['accuracy']:.4f}")
print(f"     Precision      : {best_res['precision']:.4f}")
print(f"     Recall         : {best_res['recall']:.4f}")
print(f"     F1-Score       : {best_res['f1']:.4f}")
print(f"     ROC-AUC        : {best_res['roc_auc']:.4f}")
print("=" * 60)

### 📝 Interpretasi Hasil

1. **Class Imbalance**: Dataset sangat tidak seimbang (~0.17% fraud). SMOTE berhasil menambah data sintetis kelas minoritas sehingga model lebih sensitif mendeteksi fraud.

2. **Metrik Evaluasi**: Untuk kasus fraud detection, **Recall** dan **ROC-AUC** lebih penting daripada Accuracy. Recall tinggi = model jarang melewatkan transaksi fraud.

3. **Model Comparison**:
   - Logistic Regression: cepat tapi kurang akurat untuk pola non-linear
   - Decision Tree: mudah diinterpretasi tapi rawan overfitting
   - Random Forest & Boosting: performa terbaik karena ensemble method
   - LightGBM (Tuned): model terbaik setelah Optuna optimization

4. **Optuna**: Hyperparameter tuning dengan Bayesian Optimization (TPE Sampler) berhasil meningkatkan ROC-AUC dibanding model baseline.

5. **MLFlow**: Semua eksperimen tercatat dengan rapi, memudahkan reproduksi dan perbandingan model.

---

### 🔮 Pengembangan Selanjutnya
- Coba threshold optimization untuk trade-off precision vs recall
- Tambah feature engineering berbasis time (fraud pattern over time)
- Gunakan Isolation Forest atau Autoencoder untuk anomaly detection
- Deploy model menggunakan MLFlow Model Registry